# 예제 04. Fashion-MNIST CNN 학습과 MLP 비교
빅데이터프로그래밍 · 8주차

## 목표
- Fashion-MNIST를 CNN으로 학습시킨다
- 같은 데이터로 MLP도 학습시켜 정확도를 비교한다
- 클래스별 정확도와 오류 사례를 확인한다

MNIST보다 어려운 데이터입니다. 옷·신발·가방 10종류입니다.

**런타임 > 런타임 유형 변경 > T4 GPU** 를 먼저 선택하세요.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import pandas as pd

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 데이터 준비


In [ ]:
CLASSES = ["티셔츠", "바지", "풀오버", "드레스", "코트",
           "샌들", "셔츠", "운동화", "가방", "앵클부츠"]

transform = transforms.ToTensor()
train_set = datasets.FashionMNIST("./data", train=True,  download=True, transform=transform)
test_set  = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)

print("학습:", len(train_set), "/ 시험:", len(test_set))


In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(15, 4.2))
for ax, i in zip(axes.flatten(), range(16)):
    im, lb = train_set[i]
    ax.imshow(im.squeeze(), cmap="gray"); ax.set_title(CLASSES[lb], fontsize=10); ax.axis("off")
plt.tight_layout(); plt.show()


## 2. 두 모델 작성


In [ ]:
class MLP(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, n_classes),
        )
    def forward(self, x):
        return self.net(x)


class CNN(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 28→14
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 14→7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, n_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))


for name, m in [("MLP", MLP()), ("CNN", CNN())]:
    print(f"{name}  파라미터 {sum(p.numel() for p in m.parameters()):,}개")


## 3. 공통 학습 함수


In [ ]:
loss_fn = nn.CrossEntropyLoss()

def evaluate(model):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


def train(model, epochs=8, lr=1e-3):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for epoch in range(1, epochs + 1):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = loss_fn(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        te_loss, te_acc = evaluate(model)
        hist.append((te_loss, te_acc))
        print(f"  epoch {epoch:2d}  test loss {te_loss:.4f}  acc {te_acc:.4f}")
    return model, hist


## 4. MLP 학습


In [ ]:
print("MLP 학습")
mlp, mlp_hist = train(MLP())


## 5. CNN 학습


In [ ]:
print("CNN 학습")
cnn, cnn_hist = train(CNN())


## 6. 비교표


In [ ]:
rows = []
for name, m, hist in [("MLP", mlp, mlp_hist), ("CNN", cnn, cnn_hist)]:
    rows.append({
        "모델": name,
        "파라미터 수": f"{sum(p.numel() for p in m.parameters()):,}",
        "최종 시험 손실": round(hist[-1][0], 4),
        "최종 시험 정확도": round(hist[-1][1], 4),
        "최고 정확도": round(max(h[1] for h in hist), 4),
    })
compare = pd.DataFrame(rows)
print(compare.to_string(index=False))


In [ ]:
xs = range(1, len(mlp_hist) + 1)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(xs, [h[0] for h in mlp_hist], label="MLP")
ax[0].plot(xs, [h[0] for h in cnn_hist], label="CNN")
ax[0].set_title("test loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(xs, [h[1] for h in mlp_hist], label="MLP")
ax[1].plot(xs, [h[1] for h in cnn_hist], label="CNN")
ax[1].set_title("test accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 7. 클래스별 정확도 — 어느 옷을 헷갈리는가


In [ ]:
def per_class(model):
    model.eval()
    correct = torch.zeros(10); total = torch.zeros(10)
    with torch.no_grad():
        for x, y in test_loader:
            pred = model(x.to(device)).argmax(dim=1).cpu()
            for c in range(10):
                mask = y == c
                total[c] += mask.sum()
                correct[c] += (pred[mask] == c).sum()
    return (correct / total).tolist()


df = pd.DataFrame({
    "클래스": CLASSES,
    "MLP": [round(v, 3) for v in per_class(mlp)],
    "CNN": [round(v, 3) for v in per_class(cnn)],
})
df["차이"] = (df["CNN"] - df["MLP"]).round(3)
print(df.to_string(index=False))


## 8. CNN이 틀린 사례


In [ ]:
cnn.eval()
imgs, trues, preds = [], [], []
with torch.no_grad():
    for x, y in test_loader:
        p = cnn(x.to(device)).argmax(dim=1).cpu()
        wrong = p != y
        if wrong.any():
            imgs.append(x[wrong]); trues.append(y[wrong]); preds.append(p[wrong])
        if sum(len(t) for t in trues) > 16:
            break

imgs = torch.cat(imgs); trues = torch.cat(trues); preds = torch.cat(preds)

fig, axes = plt.subplots(2, 8, figsize=(16, 4.6))
for ax, i in zip(axes.flatten(), range(16)):
    ax.imshow(imgs[i].squeeze(), cmap="gray")
    ax.set_title(f"{CLASSES[preds[i]]}\n(정답 {CLASSES[trues[i]]})", fontsize=9, color="crimson")
    ax.axis("off")
plt.tight_layout(); plt.show()


## 9. 학습된 필터 보기
학습 전 무작위였던 필터가 무엇을 찾도록 변했는지 봅니다.


In [ ]:
w = cnn.features[0].weight.data.cpu()
print("첫 Conv 층 필터:", tuple(w.shape))

fig, axes = plt.subplots(2, 8, figsize=(14, 3.6))
for ax, i in zip(axes.flatten(), range(16)):
    ax.imshow(w[i, 0], cmap="gray"); ax.set_title(f"{i}", fontsize=9); ax.axis("off")
plt.suptitle("learned filters", y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
torch.save(cnn.state_dict(), "fashion_cnn.pt")
print("저장 완료")


## 직접 해보기
1. epoch을 15로 늘리면 두 모델의 차이가 더 벌어지나요?
2. CNN의 필터 개수를 (32, 64)로 늘려 정확도를 비교하세요.
3. 클래스별 정확도에서 CNN이 가장 크게 개선한 옷은 무엇인가요?


In [ ]:
# 여기에 작성하세요
